# all-reduce-grad-sync — ex1: synchronize gradients across ranks with all_reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-grad-sync`. Running the final beacon cell reports progress against the `Distributed: all_reduce grad sync` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: gradient synchronization via `all_reduce`
The heart of data-parallel training:
```python
loss.backward()                  # each rank fills its OWN .grad tensors
for p in model.parameters():
    dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)
    p.grad /= world_size         # convert sum → mean
optimizer.step()                 # now every rank takes the SAME step
```
Without grad sync, each rank's optimizer would drift independently — after a few steps the models diverge and training collapses. After grad sync, ranks stay bit-identical (modulo cross-GPU non-determinism in NCCL).

**Why iterate parameters.** Each parameter has its own `.grad` tensor of independent shape; `all_reduce` operates on a single tensor at a time. Real DDP fuses these into 'buckets' for bandwidth efficiency — we're doing the unfused version.

### Exercise 1 — synchronize gradients across ranks with all_reduce

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.all_reduce(p.grad, SUM)` + divide-by-world_size to average gradients across ranks for every parameter in a small model, after each rank has its own `.grad` populated.
> Keywords: DDP, grad-sync, all_reduce, parameters, mean
> ```

**KCs targeted:** `loop-parameters-all-reduce-grad`, `divide-by-world-size-for-mean`

Implement `ex1_grad_sync_worker(rank, world_size, port, out_queue)`. Each rank:

1. Inits the `gloo` process group.
2. Builds a fresh model: `model = SimpleModel()` (defined inline in the test — a `nn.Module` with one `nn.Parameter` `p` initialized to `[2.0, 4.0]`).
3. Computes a rank-dependent loss so each rank gets a DIFFERENT gradient: `loss = (model.p * (rank + 1)).sum()`. Then `loss.backward()`. Each rank now has `p.grad = [rank+1, rank+1]`.
4. **Synchronizes grads across ranks:**
   ```python
   for param in model.parameters():
       dist.all_reduce(param.grad, op=dist.ReduceOp.SUM)
       param.grad /= world_size
   ```
5. Pushes `(rank, model.p.grad.tolist())` onto `out_queue`.
6. Destroys process group.

Expected: with `world_size=2`, rank 0's pre-sync grad is `[1, 1]` and rank 1's is `[2, 2]`. After sync (mean), BOTH ranks hold `[1.5, 1.5]`.

In [ ]:
def ex1_grad_sync_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    model = SimpleModel()
    loss = (model.p * (rank + 1)).sum()
    loss.backward()
    for param in model.parameters():
        dist.all_reduce(param.grad, op=dist.ReduceOp.SUM)
        param.grad /= world_size
    out_queue.put((rank, model.p.grad.tolist()))
    dist.destroy_process_group()


<details><summary>Solution</summary>

```python
def ex1_grad_sync_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    model = SimpleModel()
    loss = (model.p * (rank + 1)).sum()
    loss.backward()
    for param in model.parameters():
        dist.all_reduce(param.grad, op=dist.ReduceOp.SUM)
        param.grad /= world_size
    out_queue.put((rank, model.p.grad.tolist()))
    dist.destroy_process_group()
```

**Order matters: `backward()` → `all_reduce` → `step()`.** If you all_reduce BEFORE backward, there's no grad to reduce yet (it's `None`). If you all_reduce AFTER step, you've already moved the parameters using a rank-local gradient — divergence.

**Why `param.grad /= world_size` not `dist.all_reduce(..., op=ReduceOp.AVG)`.** `AVG` exists in newer torch (>= 1.10) for nccl, but `gloo` doesn't support it. Sum-then-divide is the portable form. ARENA's `reduce_op_mean_divide` atom is exactly this division step — the next drill drills it in isolation.

**Real DDP does this for you.** `torch.nn.parallel.DistributedDataParallel` wraps your model and hooks `backward` so grads are all_reduced as soon as each parameter's grad is computed (during backward, not after). The naïve loop in this drill is the conceptual model; DDP is the optimized production form.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()